### **FAQ's CHATBOT**

In [19]:
import random
import pickle
import numpy as np
import json
import tensorflow as tf
import nltk
from nltk.stem import WordNetLemmatizer
'''
# Download required NLTK data
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
'''

"\n# Download required NLTK data\nnltk.download('punkt_tab')\nnltk.download('wordnet')\nnltk.download('omw-1.4')\n"

In [21]:
#Loading the intents file
lemmatizer = WordNetLemmatizer()

intents = json.loads(open('Inteents.json').read())

words = []
classes = []
documents = []
ignore_letters = ['.',',','!','?']

In [22]:
for intent in intents['intents']:
    for pattern in intent['patterns']:
        wordList = nltk.word_tokenize(pattern)
        words.extend(wordList)
        documents.append((wordList,intent['tag']))
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

words = [lemmatizer.lemmatize(word) for word in words if word not in ignore_letters]
words = sorted(set(words))

classes = sorted(set(classes))

In [23]:
# saving the class and word model
pickle.dump(words,open('words.pkl','wb'))
pickle.dump(classes,open('classes.pkl','wb'))

In [24]:
training = []
outputEmpty = [0] * len(classes)

for document in documents:
    bag = []
    wordPatterns = document[0]
    wordPatterns = [lemmatizer.lemmatize(word.lower()) for word in wordPatterns]
    for word in words: bag.append(1) if word in wordPatterns else bag.append(0)
    
    outputRow = list(outputEmpty)
    outputRow[classes.index(document[1])] = 1
    training.append(bag + outputRow)

random.shuffle(training)
training = np.array(training)

trainX = training[:, :len(words)]
trainY = training[:, len(words):]

### Building the model


In [25]:
model = tf.keras.Sequential()

model.add(tf.keras.layers.Dense(128, input_shape = (len(trainX[0]),),activation = 'relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(64,activation = 'relu'))
model.add(tf.keras.layers.Dense(len(trainY[0]),activation = 'softmax'))

c:\Users\levie\miniconda3\envs\geo\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Optimizers

In [26]:
sgd = tf.keras.optimizers.SGD(learning_rate= 0.01, momentum = 0.9, nesterov= True)
model.compile(loss = 'categorical_crossentropy',optimizer = sgd, metrics = ['accuracy'])

### Training the model

In [27]:
mod = model.fit(np.array(trainX),np.array(trainY),epochs = 200,batch_size= 5, verbose = 1)

model.save('chatbot_FAQ.h5',mod)
print("Executed successfully")

Epoch 1/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 21s 16ms/step - accuracy: 0.0280 - loss: 4.3953
Epoch 2/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.0429 - loss: 4.2962
Epoch 3/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0575 - loss: 4.2027
Epoch 4/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.0455 - loss: 4.0055
Epoch 5/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.0856 - loss: 3.8716
Epoch 6/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.1688 - loss: 3.6884
Epoch 7/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.2554 - loss: 3.3958
Epoch 8/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.1738 - loss: 3.2763
Epoch 9/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.2379 - loss: 2.8061
Epoch 10/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.2818 - loss: 2.7359
Epoch 11/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.3017 - loss: 2.5266
Epoch 12/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/ste

Executed successfully
